# Phase 3: High-Frequency Detail GAN Training (Production Architecture)

**Objective:** Train an adversarial synthesis network to generate high-frequency UV displacement maps containing subject-specific micro-wrinkles and pore texture, conditioned on coarse neutral geometry and multi-view features.

### Core Architectural Features:
1. **U-Net Generator:** InstanceNorm2d (affine=True), multi-view cross-attention bottleneck with LayerNorm & residual connection, AdaIN style modulation from identity/expression codes.
2. **PatchGAN Discriminator:** Spectral Normalization with 70×70 receptive field patches.
3. **Lazy R1 Gradient Penalty:** Computed every 16 steps strictly in **FP32** accumulating into discriminator gradients.
4. **Masked Reconstruction Loss:** L1 loss restricted strictly to valid facial UV pixels (`mask == 1.0`), annealed linearly.
5. **Hardware & Distributed Strategy:** Dynamic GPU topology detection (`torch.cuda.device_count()`), PyTorch DDP (`torchrun`), PyTorch AMP mixed-precision float16.
6. **Quota Guard & Checkpoints:** Sliding window of 2 step checkpoints, `checkpoint_latest.pt` (full optimizer/scaler state), and continuous EMA weights in `ema_generator.pt`.


In [ ]:
# Cell 1: Environment & Repository Setup
import os
import sys
import subprocess
from pathlib import Path

print("--- Setting Up Humanoid-Face-3D Workspace ---")
target_dir = Path("/kaggle/working/Humanoid-Face-3D")
if not (target_dir / "src/pipeline.py").exists():
    subprocess.run(["git", "clone", "--recurse-submodules", "https://github.com/NetPranav/Humanoid-Face-3D.git", str(target_dir)], check=True)
os.chdir(str(target_dir))
subprocess.run(["git", "fetch", "origin", "main"], check=False)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=False)
subprocess.run(["git", "submodule", "update", "--init", "--recursive"], check=False)

if str(target_dir) not in sys.path:
    sys.path.insert(0, str(target_dir))
if "." not in sys.path:
    sys.path.insert(0, ".")
print("Active working directory:", os.getcwd())


In [ ]:
# Cell 2: Install Runtime Dependencies & Detect GPU Topology
import subprocess
subprocess.run(["pip", "install", "trimesh", "opencv-python", "pyyaml", "scipy", "Pillow", "matplotlib", "--quiet"], check=True)

import torch
print(f"PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "Critical Error: CUDA GPU accelerator required for GAN training!"
n_gpus = torch.cuda.device_count()
print(f"Detected {n_gpus} GPU(s) for training:")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  [GPU {i}] {torch.cuda.get_device_name(i)} | Memory: {props.total_memory / 1e9:.2f} GB")


In [ ]:
# Cell 3: Dataset Discovery & Demographic Preprocessing Assurance
import json
import shutil
import subprocess
from pathlib import Path

print("--- Locating / Preparing UV Displacement Dataset ---")
DATA_DIR = None

# Check /kaggle/input for existing preprocessed dataset
for cand in list(Path("/kaggle/input").glob("**/uv_displacement_dataset_1024")) + list(Path("/kaggle/input").glob("**/normalization_stats.json")):
    p = cand if cand.is_dir() else cand.parent
    if list(p.glob("*_disp.png")):
        DATA_DIR = p
        print(f"Found mounted preprocessed dataset at: {DATA_DIR}")
        break

# If not mounted in /kaggle/input, generate 20-subject demographic corpus inside session
if DATA_DIR is None:
    print("Preprocessed dataset not found in /kaggle/input. Running high-throughput demographic synthesis engine...")
    flame_pkl_candidates = list(Path("/kaggle/input").glob("**/generic_model.pkl")) + list(Path("data/flame_model").glob("generic_model.pkl"))
    if not flame_pkl_candidates:
        raise FileNotFoundError("generic_model.pkl not found! Please attach flame-model dataset.")
    flame_pkl = flame_pkl_candidates[0]
    
    template_candidates = list(Path("/kaggle/input").glob("**/head_template.obj")) + list(Path("data/flame_model").glob("head_template.obj"))
    template_p = template_candidates[0] if template_candidates else None
    
    DATA_DIR = Path("/tmp/uv_displacement_dataset_1024")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    
    synth_cmd = [
        "python", "scripts/build_uv_displacement_dataset.py",
        "--synthesize_demographic_corpus", "20",
        "--flame_model", str(flame_pkl),
        "--output_dir", str(DATA_DIR),
        "--resolution", "1024"
    ]
    if template_p:
        synth_cmd.extend(["--uv_template", str(template_p)])
        
    print("Executing synthesis command:", " ".join(synth_cmd))
    subprocess.run(synth_cmd, check=True)

disp_files = list(DATA_DIR.glob("*_disp.png"))
print(f"Verified dataset at {DATA_DIR} containing {len(disp_files)} displacement maps.")
assert len(disp_files) >= 2, f"Expected >= 2 displacement samples, found {len(disp_files)}"

stats_file = DATA_DIR / "normalization_stats.json"
if stats_file.exists():
    with open(stats_file) as f:
        stats = json.load(f)
    print(f"Loaded normalization stats: p99 = {stats.get('p99_mm', 'N/A')} mm, resolution = {stats.get('resolution', 'N/A')}")


In [ ]:
# Cell 4: Launch Detail GAN Training with Quota Guards & Resumability
import os
import subprocess
import torch
from pathlib import Path

n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
checkpoint_dir = Path("checkpoints/stage3_detail")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

train_cmd = [
    "torchrun", f"--nproc_per_node={n_gpus}",
    "src/stage3_detail/trainer.py",
    "--data_dir", str(DATA_DIR),
    "--checkpoint_dir", str(checkpoint_dir),
    "--resolution", "512",
    "--checkpoint_every", "250",
    "--total_steps", "1500",
    "--batch_size", "4"
]

env = os.environ.copy()
env["PYTHONPATH"] = f"{os.getcwd()}:{env.get('PYTHONPATH', '')}"

print("--- Launching Detail GAN Training ---")
print("Command:", " ".join(train_cmd))
subprocess.run(train_cmd, env=env, check=True)
print("Training execution completed successfully.")


In [ ]:
# Cell 5: Validation Evaluation & Batch Output Diversity Gate
import torch
import cv2
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.stage3_detail.generator import DetailGenerator
from src.stage3_detail.data import UVDisplacementDataset

print("\n--- Phase 3 Detail GAN Evaluation & Diversity Gate ---")
ckpt_ema = Path("checkpoints/stage3_detail/ema_generator.pt")
assert ckpt_ema.exists(), f"Critical Invariant Failure: Expected {ckpt_ema} to exist after training!"

device = "cuda" if torch.cuda.is_available() else "cpu"
gen = DetailGenerator().to(device)
try:
    state_dict = torch.load(ckpt_ema, map_location=device, weights_only=False)
except TypeError:
    state_dict = torch.load(ckpt_ema, map_location=device)
gen.load_state_dict(state_dict)
gen.eval()

val_ds = UVDisplacementDataset(str(DATA_DIR), is_train=False, target_resolution=512)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=min(4, len(val_ds)), shuffle=False)

with torch.no_grad():
    batch = next(iter(val_loader))
    pos = batch["pos"].to(device)
    norm = batch["norm"].to(device)
    feats = batch["per_view_feats"].to(device)
    beta = batch["beta"].to(device)
    psi = batch["psi"].to(device)
    real_disp = batch["disp"].to(device)
    mask = batch["mask"].to(device)

    pred_disp = gen(pos, norm, feats, beta, psi)
    batch_std = float(pred_disp.std().item())
    pred_min = float(pred_disp.min().item())
    pred_max = float(pred_disp.max().item())

    print(f"Output Shape:             {pred_disp.shape}")
    print(f"Displacement Range:       [{pred_min:.4f}, {pred_max:.4f}]")
    print(f"Batch Standard Deviation: {batch_std:.4f} (Required Gate: > 0.010)")

    # Gate 1: Spatial Diversity (Anti-Mode Collapse)
    assert batch_std > 0.010, f"GATE FAILURE: Mode collapse detected (std={batch_std:.4f} <= 0.010)"
    print("GATE 1 PASSED: Detail generator produces diverse spatial micro-displacements with zero mode collapse.")

    # Copy normalization stats to checkpoint dir
    stats_src = DATA_DIR / "normalization_stats.json"
    if stats_src.exists():
        shutil.copy(stats_src, "checkpoints/stage3_detail/normalization_stats.json")

    # Generate visual validation preview
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    disp_pred_np = pred_disp[0, 0].cpu().numpy()
    disp_real_np = real_disp[0, 0].cpu().numpy()
    mask_np = mask[0, 0].cpu().numpy()

    im0 = axes[0].imshow(disp_pred_np, cmap="inferno", vmin=-1.0, vmax=1.0)
    axes[0].set_title(f"Synthesized Displacement (std={batch_std:.4f})")
    axes[0].axis("off")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(disp_real_np, cmap="inferno", vmin=-1.0, vmax=1.0)
    axes[1].set_title("Target Ground Truth Disp")
    axes[1].axis("off")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    axes[2].imshow(mask_np, cmap="gray")
    axes[2].set_title("UV Boundary Mask")
    axes[2].axis("off")

    plt.tight_layout()
    preview_path = Path("checkpoints/stage3_detail/validation_preview.png")
    plt.savefig(preview_path, dpi=150)
    plt.close()
    print(f"Saved validation comparison preview to: {preview_path}")


In [ ]:
# Cell 6: Package & Archive Output Assets
import tarfile
from pathlib import Path

print("\n--- Packaging Trained Detail GAN Model & Checkpoints ---")
output_archive = Path("/kaggle/working/detail_gan_assets.tar.gz")
ckpt_dir = Path("checkpoints/stage3_detail")

with tarfile.open(output_archive, "w:gz") as tar:
    for f in ckpt_dir.glob("*"):
        if f.is_file():
            tar.add(f, arcname=f.name)
            print(f"  Archived: {f.name} ({f.stat().st_size / 1e6:.2f} MB)")

print(f"\nAll assets successfully packaged to: {output_archive} ({output_archive.stat().st_size / 1e6:.2f} MB).")
